# 01b — Data Preparation for validation set

This notebook looks at the raw NMRshiftdb2 data (https://sourceforge.net/projects/nmrshiftdb2/files/data/) and cleaned it up for processing as well as make is ready for use downstream.

## Imports

In [1]:
import re

import pandas as pd
import rdkit
from rdkit import Chem
from tqdm import tqdm

from dept_similarity.constants import PROCESSED_DATA_DIR, RAW_DATA_DIR
from dept_similarity.utils import get_spectrum_score, passed_cleanup

rdkit.RDLogger.DisableLog("rdApp.*")  # Disable RDKit warnings

pd.set_option("mode.chained_assignment", None)  # Disable chained assignment warnings
tqdm.pandas()

# Loading and cleaning the NMRShiftDB

In [2]:
# Read the SDF file and convert it to a DataFrame
sdf_file = f"{RAW_DATA_DIR}/nmrshiftdb2withsignals.sd"
supplier = Chem.SDMolSupplier(sdf_file, removeHs=False)

In [3]:
data = []

for mol in tqdm(supplier):
    if mol is None:
        continue

    props = mol.GetPropsAsDict()

    if props.get("INChI key") == "null":
        continue

    props["SMILES"] = Chem.MolToSmiles(mol)
    data.append(props)

library_df = pd.DataFrame(data)
library_df.head(2)

100%|██████████| 58187/58187 [00:06<00:00, 8675.87it/s] 


,INChI key,INChI,Temperature [K],nmrshiftdb2 ID,Field Strength [MHz],Spectrum 13C 0,Solvent,SMILES,NMRBasisSet,Program,...,Spectrum DEPTQ 12,rawdata DEPTQ 12,rawdata Cyclic voltammetry 1,Spectrum Cyclic voltammetry 1,Machine,rawdata 19F 2,"Spectrum H,C-HMQC 6","rawdata H,C-HMQC 6","Spectrum H,C-HMQC 5","rawdata H,C-HMQC 5"
0,QGMWDUUHVVLHNP-JQXSQYPDSA-N,InChI=1S/C15H22O3/c1-13(2)7-4-8-14(3)12(13)6-5...,0:298,234.0,0:50,17.6;0.0Q;10|18.3;0.0T;0|22.6;0.0Q;12|26.5;0.0...,0:Chloroform-D1 (CDCl3),CC1(C)CCC[C@@]2(C)C1CC=C(C=O)[C@]2(O)C=O,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GJNXTRPTZOVADS-PWRGDLIESA-N,InChI=1S/C15H24O/c1-10-4-7-15-11(2)12(9-16)8-1...,0:298,2151.0,0:125,17.7;0.0Q;10|20.0;0.0Q;12|22.9;0.0Q;14|28.9;0....,0:Chloroform-D1 (CDCl3),C[C@@H]1C(CO)=C[C@@]2(C)CC[C@@H]3[C@@H](C)CC[C...,2: 3:,"1:ACD/Labs C+H NMR Predictors and DB, 2020.1.0",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
len(library_df), library_df["INChI key"].nunique()

(57325, 57205)

## Filtering out predicted spectra

In [5]:
filtered_subset = library_df[
    library_df["Solvent"].notna()
]  # Solvent includes the solvent used for the experimental measurement

filtered_subset = filtered_subset[
    filtered_subset["Field Strength [MHz]"].notna()
]  # Field Strength includes the magnetic field strength used for the experimental measurement

print(f"Number of spectra with solvent and field strength information: {len(filtered_subset)}")

filtered_subset = filtered_subset[
    filtered_subset["Program"].isna()
]  # Program includes the software used for predicted structures

filtered_subset = filtered_subset[
    filtered_subset["NMRProgram"].isna()
]  # NMRProgram includes the software used for predicted structures

filtered_subset.head(2)

Number of spectra with solvent and field strength information: 57313


,INChI key,INChI,Temperature [K],nmrshiftdb2 ID,Field Strength [MHz],Spectrum 13C 0,Solvent,SMILES,NMRBasisSet,Program,...,Spectrum DEPTQ 12,rawdata DEPTQ 12,rawdata Cyclic voltammetry 1,Spectrum Cyclic voltammetry 1,Machine,rawdata 19F 2,"Spectrum H,C-HMQC 6","rawdata H,C-HMQC 6","Spectrum H,C-HMQC 5","rawdata H,C-HMQC 5"
0,QGMWDUUHVVLHNP-JQXSQYPDSA-N,InChI=1S/C15H22O3/c1-13(2)7-4-8-14(3)12(13)6-5...,0:298,234.0,0:50,17.6;0.0Q;10|18.3;0.0T;0|22.6;0.0Q;12|26.5;0.0...,0:Chloroform-D1 (CDCl3),CC1(C)CCC[C@@]2(C)C1CC=C(C=O)[C@]2(O)C=O,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,UNPSXBKCCZEIRN-UHFFFAOYSA-N,"InChI=1S/C8H16N2O/c1-7(2)6-10(11)8(3,4)9(7)5/h...",0:298,2189.0,0:50.328,23.3;0.0Q;8|23.3;0.0Q;9|23.5;0.0Q;5|23.5;0.0Q;...,0:Acetone-D6 ((CD3)2CO),CN1C(C)(C)C=[N+]([O-])C1(C)C,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Relevant columns for DEPT similarity analysis
columns_to_keep = [
    "INChI key",
    "Temperature [K]",
    "nmrshiftdb2 ID",
    "Field Strength [MHz]",
    "Solvent",
    "SMILES",
    "Program",
    "NMRMethod",
]
dept_columns = [
    col for col in filtered_subset.columns if "Spectrum DEPT135" in col or "Spectrum 13C" in col
]
columns_to_keep.extend(dept_columns)

filtered_df = filtered_subset[columns_to_keep]
print(f"Number of spectra with DEPT135 or 13C information: {len(filtered_df)}")

Number of spectra with DEPT135 or 13C information: 53042


In [7]:
# Count the number of spectra for each INChI key
filtered_df["spectrum_count"] = filtered_df.apply(
    lambda row: sum([1 for col in dept_columns if pd.notna(row[col])]), axis=1
)

# Drop columns with no DEPT based spectra
filtered_df = filtered_df[filtered_df["spectrum_count"] > 0].copy()

In [8]:
len(filtered_df), filtered_df["INChI key"].nunique()

(29075, 29024)

## Selecting the most representative DEPT-like spectra in case multiple exist

In [9]:
# Creating a long-format table with each row as unique spectra


def get_metadata_dict(metadata_str):
    if pd.isna(metadata_str):
        return {}

    combined_values = re.findall(r"(\d+):(.*?)(?=\s+\d+:|$)", metadata_str)
    temp_dict = {int(temp): value.lstrip().rstrip() for temp, value in combined_values}

    return temp_dict


melted_table = []
for _, row in tqdm(filtered_df.iterrows(), total=filtered_df.shape[0]):
    # Skip entries without SMILES
    if pd.isna(row["SMILES"]):
        continue

    temp_dict = get_metadata_dict(row["Temperature [K]"])
    field_dict = get_metadata_dict(row["Field Strength [MHz]"])
    solvent_dict = get_metadata_dict(row["Solvent"])
    program_dict = get_metadata_dict(row["Program"])
    nmrmethod_dict = get_metadata_dict(row["NMRMethod"])

    for spectrum_col in dept_columns:
        if pd.notna(row[spectrum_col]):
            spectrum_index = spectrum_col.replace("Spectrum DEPT135", "").replace(
                "Spectrum 13C", ""
            )

            # Make spectrum id as integer
            spectrum_index = (
                int(re.findall(r"\d+", spectrum_index)[0])
                if re.findall(r"\d+", spectrum_index)
                else None
            )

            # Skip predicted spectra: those with a Program or NMRMethod set for this index
            if spectrum_index in program_dict or spectrum_index in nmrmethod_dict:
                continue

            melted_table.append(
                {
                    "compound_id": (
                        "nmrshiftdb2_" + str(int(row["nmrshiftdb2 ID"]))
                        if pd.notna(row["nmrshiftdb2 ID"])
                        else None
                    ),
                    "inchikey": row["INChI key"],
                    "smiles": row["SMILES"],
                    "spectrum": row[spectrum_col],
                    "temperature_k": temp_dict.get(spectrum_index, None),
                    "field_strength_mhz": field_dict.get(spectrum_index, None),
                    "solvent": solvent_dict.get(spectrum_index, None),
                    "spectrum_type": "DEPT135" if "DEPT135" in spectrum_col else "13C",
                    "spectrum_count": row["spectrum_count"],
                }
            )

melted_df = pd.DataFrame(melted_table)
melted_df.head(2)

100%|██████████| 29075/29075 [00:01<00:00, 23006.84it/s]


,compound_id,inchikey,smiles,spectrum,temperature_k,field_strength_mhz,solvent,spectrum_type,spectrum_count
0,nmrshiftdb2_234,QGMWDUUHVVLHNP-JQXSQYPDSA-N,CC1(C)CCC[C@@]2(C)C1CC=C(C=O)[C@]2(O)C=O,17.6;0.0Q;10|18.3;0.0T;0|22.6;0.0Q;12|26.5;0.0...,298,50,Chloroform-D1 (CDCl3),13C,1
1,nmrshiftdb2_2189,UNPSXBKCCZEIRN-UHFFFAOYSA-N,CN1C(C)(C)C=[N+]([O-])C1(C)C,23.3;0.0Q;8|23.3;0.0Q;9|23.5;0.0Q;5|23.5;0.0Q;...,298,50.328,Acetone-D6 ((CD3)2CO),13C,1


## Build DEPT spectra

In [10]:
def build_dept_like_spectra(spectrum_str):
    """
    Convert the C13 spectrum into DEPT135-like spectrum.

    Args:
        spectrum_str (str): C13 spectrum string in the format "shift;mult;intensity|shift;mult;intensity|..."

    Returns:
        list: List of signed chemical shifts.
    """
    cppm_signed_peaks = []

    spectra_parts = spectrum_str.split("|")
    for peak in spectra_parts:
        if not peak.strip():
            continue  # Skip empty parts

        shift, mult, _ = peak.split(";")
        label = mult[-1].lower()  # S, D, T, Q

        if label == "s":
            continue  # Skip singlet/quaternary C as they are not observed in DEPT135 spectra

        signed_shift = float(shift) * (
            -1 if label == "t" else 1
        )  # CH2 peaks are negative in DEPT135 spectra
        cppm_signed_peaks.append(signed_shift)

    return cppm_signed_peaks

In [11]:
melted_df["signed_shifts"] = melted_df.progress_apply(
    lambda row: build_dept_like_spectra(row["spectrum"]), axis=1
)

100%|██████████| 30282/30282 [00:00<00:00, 193732.38it/s]


In [12]:
# number of peaks in each spectrum for score
melted_df["score"] = melted_df.progress_apply(get_spectrum_score, axis=1)
melted_df["inchikey14"] = melted_df["inchikey"].str[:14]

100%|██████████| 30282/30282 [00:00<00:00, 168061.42it/s]


In [13]:
melted_df.to_parquet("../data/processed/full_nmrshiftdb_dept135_like_spectra.parquet", index=False)

## Collapsing each inchikey compound pair to single spectra

In [14]:
tmp_filtered_df = melted_df[
    ~melted_df["temperature_k"].str.contains("Unreported|Unknown", case=False, na=False)
].copy()
print(
    f"Temperature filtered:\n"
    f"  Final count: {tmp_filtered_df.shape[0]} spectra and {tmp_filtered_df['inchikey14'].nunique()} unique compounds."
)

field_filtered_df = tmp_filtered_df[
    ~tmp_filtered_df["field_strength_mhz"].str.contains("Unreported|Unknown", case=False, na=False)
].copy()
print(
    f"Field strength filtered:\n"
    f"  Final count: {field_filtered_df.shape[0]} spectra and {field_filtered_df['inchikey14'].nunique()} unique compounds."
)

solvent_filtered_df = field_filtered_df[
    ~field_filtered_df["solvent"].str.contains("Unreported|Unknown", case=False, na=False)
].copy()
print(
    f"Solvent filtered:\n"
    f"  Final count: {solvent_filtered_df.shape[0]} spectra and {solvent_filtered_df['inchikey14'].nunique()} unique compounds."
)

Temperature filtered:
  Final count: 1944 spectra and 1831 unique compounds.
Field strength filtered:
  Final count: 1437 spectra and 1360 unique compounds.
Solvent filtered:
  Final count: 1436 spectra and 1359 unique compounds.


In [15]:
# Select the best spectrum for each INChI key based on the score
best_spectra_df = (
    solvent_filtered_df.sort_values("score", ascending=False)
    .groupby(["inchikey14", "compound_id"])
    .first()
    .reset_index()
)
best_spectra_df.head(2)

,inchikey14,compound_id,inchikey,smiles,spectrum,temperature_k,field_strength_mhz,solvent,spectrum_type,spectrum_count,signed_shifts,score
0,AANRHGFZHQCTAB,nmrshiftdb2_60004828,AANRHGFZHQCTAB-UHFFFAOYSA-N,CC(C)(N=[N+]=[N-])C(=O)N(Cc1ccccc1)Cc1ccccc1,126.8;0.0D;1|126.8;0.0D;2|127.9;0.0D;11|127.9;...,298,100.0,Chloroform-D1 (CDCl3),13C,1,"[126.8, 126.8, 127.9, 127.9, -50.6, -48.6, 25....",290
1,AATMCCVYMZDTCD,nmrshiftdb2_10008594,AATMCCVYMZDTCD-UHFFFAOYSA-N,C1=Cc2ccccc2C=CC1,26.6;0.0T;6|125.9;0.0D;0|125.9;0.0D;5|127.7;0....,313,23,TETRACHLORO-METHANE (CCl4),13C,2,"[-26.6, 125.9, 125.9, 127.7, 127.7, 130.3, 130...",240


In [16]:
inchikey14_compound_counts = (
    melted_df.groupby("inchikey14")["compound_id"].nunique().reset_index(name="compound_count")
)

multiple_compounds_per_inchikey14 = inchikey14_compound_counts[
    inchikey14_compound_counts["compound_count"] > 1
]

print(
    f"Number of inchikey14 groups with multiple compounds: {len(multiple_compounds_per_inchikey14)}"
)

Number of inchikey14 groups with multiple compounds: 630


In [17]:
len(best_spectra_df), len(best_spectra_df["inchikey"].unique())

(1374, 1371)

In [18]:
len(best_spectra_df), len(best_spectra_df["inchikey"].unique())

(1374, 1371)

In [19]:
best_spectra_df["passed_cleanup"] = best_spectra_df["smiles"].progress_apply(passed_cleanup)

100%|██████████| 1374/1374 [00:00<00:00, 11918.72it/s]


In [20]:
best_spectra_df["passed_cleanup"].value_counts()

passed_cleanup
True     1171
False     203
Name: count, dtype: int64

In [21]:
experimental_df = best_spectra_df[
    [
        "compound_id",
        "inchikey",
        "inchikey14",
        "smiles",
        "signed_shifts",
        "passed_cleanup",
    ]
]

In [22]:
experimental_df["dept_intensity"] = (
    None  # Placeholder for DEPT intensity, as we are only using the signed and binary intensities for validation
)

In [23]:
experimental_df.to_parquet("../data/processed/nmrshiftdb_dept135_like_spectra.parquet", index=False)

# Identifying overlap between the experimental and library spectra

In [24]:
library_df = pd.read_parquet(f"{PROCESSED_DATA_DIR}/library_data.parquet")
lib_inchikey14_set = set(library_df["inchikey14"].unique())
library_df.head(2)

,compound_id,smiles,signed_shifts,inchikey,inchikey14,passed_cleanup
0,dept_1000000_CVVMLCYWLCXJCW,CC1=CC(O)=CC(O)=C1O[C@@H]1O[C@H](CO)[C@@H](O)[...,"[16.31, -62.06, 70.91, 74.96, 77.43, 77.72, 10...",CVVMLCYWLCXJCW-HMUNZLOLSA-N,CVVMLCYWLCXJCW,True
1,dept_1000001_DTKRGJZNRMIPRF,O=C(OC[C@@H](O)[C@@H](O)[C@H](O)[C@H](O)CO)C1=...,"[-64.36, -66.81, 71.31, 72.41, 72.49, 72.59, 1...",DTKRGJZNRMIPRF-DDHJBXDOSA-N,DTKRGJZNRMIPRF,True


In [25]:
experimental_df_subset = experimental_df[experimental_df["passed_cleanup"]].copy()
overlapped_experimental_df = experimental_df_subset[
    experimental_df_subset["inchikey14"].isin(lib_inchikey14_set)
]
overlapped_experimental_df.head(2)

,compound_id,inchikey,inchikey14,smiles,signed_shifts,passed_cleanup,dept_intensity
1,nmrshiftdb2_10008594,AATMCCVYMZDTCD-UHFFFAOYSA-N,AATMCCVYMZDTCD,C1=Cc2ccccc2C=CC1,"[-26.6, 125.9, 125.9, 127.7, 127.7, 130.3, 130...",True,None
2,nmrshiftdb2_60004200,AAWKCAAKYFTATH-STJFUXCQSA-N,AAWKCAAKYFTATH,C=C/C(C)=C/C[C@H]1C(C)=CCC2C(C)(C)CCC[C@@]21C,"[22.2, 33.51, 14.24, 22.72, -19.09, -26.5, -23...",True,None


In [26]:
len(overlapped_experimental_df), len(overlapped_experimental_df["inchikey14"].unique())

(648, 644)

## Save the cleaned datasets

In [27]:
overlapped_experimental_df.to_parquet(
    f"{PROCESSED_DATA_DIR}/experimental_validation_set.parquet",
    index=False,
    engine="pyarrow",
)

In [28]:
multiple_spectra_overlap_df = solvent_filtered_df.copy()
multiple_spectra_overlap_df = multiple_spectra_overlap_df[
    multiple_spectra_overlap_df["inchikey14"].isin(lib_inchikey14_set)
].copy()
len(multiple_spectra_overlap_df), len(multiple_spectra_overlap_df["inchikey14"].unique())

(681, 648)

In [29]:
multiple_spectra_overlap_df.to_parquet(
    f"{PROCESSED_DATA_DIR}/multiple_spectra_overlap_set.parquet",
    index=False,
    engine="pyarrow",
)